In [1]:
import pandas as pd
import numpy as np

tkpi = pd.read_csv("tkpi_with_category.csv")
urt = pd.read_csv("urt_with_category.csv")



In [4]:
tkpi.columns

Index(['food_code', 'food_name', 'source', 'water_g_100g', 'energy_kcal_100g',
       'protein_g_100g', 'fat_g_100g', 'carb_g_100g', 'fiber_g_100g',
       'ash_g_100g', 'calcium_mg_100g', 'phosphorus_mg_100g', 'iron_mg_100g',
       'sodium_mg_100g', 'potassium_mg_100g', 'copper_mg_100g', 'zinc_mg_100g',
       'retinol_mcg_100g', 'beta_carotene_mcg_100g', 'carotene_total_mcg_100g',
       'thiamin_mg_100g', 'riboflavin_mg_100g', 'niacin_mg_100g',
       'vitamin_c_mg_100g', 'bdd_percent', 'tkpi_main_group',
       'tkpi_group_code', 'category_code', 'category_name', 'category_source',
       'food_name_clean', 'food_name_match'],
      dtype='object')

In [5]:
urt.columns


Index(['food_name', 'weight_gram', 'urt', 'category_code', 'category_name',
       'urt_food_name_clean', 'urt_food_name_match'],
      dtype='object')

In [7]:
import pandas as pd

# tkpi = your TKPI dataframe
# urt = your URT dataframe

# ============================================================
# 1. Prepare merge keys
# ============================================================

tkpi_merge = tkpi.copy()
urt_merge = urt.copy()

tkpi_merge["category_code"] = (
    tkpi_merge["category_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

urt_merge["category_code"] = (
    urt_merge["category_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

tkpi_merge["food_name_match"] = (
    tkpi_merge["food_name_match"]
    .astype(str)
    .str.strip()
    .str.lower()
)

urt_merge["urt_food_name_match"] = (
    urt_merge["urt_food_name_match"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# 2. Prepare URT columns for merge
# ============================================================

urt_for_merge = urt_merge.rename(
    columns={
        "urt_food_name_match": "food_name_match",
        "weight_gram": "gram_per_portion"
    }
)

urt_for_merge = urt_for_merge[
    [
        "category_code",
        "food_name_match",
        "urt",
        "gram_per_portion"
    ]
].copy()

# Remove duplicate merge keys if any
urt_for_merge = urt_for_merge.drop_duplicates(
    subset=["category_code", "food_name_match"],
    keep="first"
)

# ============================================================
# 3. Merge TKPI + URT
# ============================================================

merged = tkpi_merge.merge(
    urt_for_merge,
    on=["category_code", "food_name_match"],
    how="left"
)

# ============================================================
# 4. Check result
# ============================================================

print("Total TKPI rows:", len(tkpi_merge))
print("Total merged rows:", len(merged))

print("\nMerged preview:")
print(
    merged[
        [
            "food_code",
            "food_name",
            "category_code",
            "food_name_match",
            "urt",
            "gram_per_portion"
        ]
    ].head(20)
)

# ============================================================
# 5. Save result
# ============================================================



Total TKPI rows: 1145
Total merged rows: 1145

Merged preview:
   food_code                            food_name category_code  \
0      AR001                 Beras giling, mentah            MP   
1      AR002      Beras giling var pelita, mentah            MP   
2      AR003    Beras giling var rojolele, mentah            MP   
3      AR004                  Beras hitam, mentah            MP   
4      AR005  Beras jagung kuning, kering, mentah            MP   
5      AR006   Beras jagung putih, kering, mentah            MP   
6      AR007     Beras ketan hitam tumbuk, mentah            MP   
7      AR008     Beras ketan putih tumbuk, mentah            MP   
8      AR009                 Beras ladang, mentah            MP   
9      AR010                  Beras menir, mentah            MP   
10     AR011                      Beras parboiled            MP   
11     AR012                 Beras tumbuk, mentah            MP   
12     AR013           Beras tumbuk merah, mentah            MP   

In [8]:
merged.category_code.value_counts()

category_code
LH     310
MP     241
S      216
LN     135
B      121
NAN     52
G       34
M       19
SS      17
Name: count, dtype: int64